In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Load metric.csv
metric_fp = "metric.csv"
df_metric = pd.read_csv(metric_fp)

# Ensure correct dtypes
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Timezone: Asia/Shanghai (UTC+8) per instructions
tz = pytz.timezone('Asia/Shanghai')

# Overall min/max timestamps (as timezone-aware datetimes)
overall_min_ts = int(df_metric['timestamp'].min())
overall_max_ts = int(df_metric['timestamp'].max())
overall_min_dt = pd.to_datetime(overall_min_ts, unit='s', utc=True).tz_convert(tz)
overall_max_dt = pd.to_datetime(overall_max_ts, unit='s', utc=True).tz_convert(tz)
overall_range = pd.DataFrame({
    'overall_min_timestamp': [overall_min_ts],
    'overall_min_datetime_ASIA_SHANGHAI': [overall_min_dt],
    'overall_max_timestamp': [overall_max_ts],
    'overall_max_datetime_ASIA_SHANGHAI': [overall_max_dt]
})

# Aggregate per (cmdb_id, kpi_name)
agg_df = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_points = ('value', 'count'),
        min_timestamp = ('timestamp', 'min'),
        max_timestamp = ('timestamp', 'max'),
        mean_value = ('value', 'mean'),
        median_value = ('value', 'median'),
        std_value = ('value', 'std'),
        p95_value = ('value', lambda x: x.quantile(0.95))
    )
)

# Convert min/max timestamps to timezone-aware datetimes (Asia/Shanghai) for readability
agg_df['min_datetime_ASIA_SHANGHAI'] = pd.to_datetime(agg_df['min_timestamp'], unit='s', utc=True).dt.tz_convert(tz)
agg_df['max_datetime_ASIA_SHANGHAI'] = pd.to_datetime(agg_df['max_timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Keep the full aggregated table in kernel memory for later steps (agg_df)
# Prepare compact display: sort by count desc and limit to top 20 rows, select key columns
agg_top20 = (
    agg_df
    .sort_values('count_points', ascending=False)
    .loc[:, ['cmdb_id','kpi_name','count_points','min_timestamp','min_datetime_ASIA_SHANGHAI',
             'max_timestamp','max_datetime_ASIA_SHANGHAI','mean_value','median_value','std_value','p95_value']]
    .head(20)
)

# Display the compact results (variables as output)
agg_top20, overall_range

```
Out[1]:
```
# Build a concise plain-English summary using previously computed results (agg_top20, overall_range, agg_df exist in kernel)
min_dt = overall_range.loc[0, 'overall_min_datetime_ASIA_SHANGHAI']
max_dt = overall_range.loc[0, 'overall_max_datetime_ASIA_SHANGHAI']

lines = []
lines.append("Summary of metric.csv aggregation:")
lines.append(f"- Overall timestamp range (Asia/Shanghai): {min_dt} to {max_dt}.")
lines.append("- Aggregation performed per (cmdb_id, kpi_name): count, min/max timestamp, mean, median, std, and global P95.")
lines.append("- Full aggregated table is stored in variable `agg_df`. A compact top-20 view is in `agg_top20`.")
lines.append("- Top entries by point count (showing up to 5):")

for i, row in agg_top20.head(5).reset_index(drop=True).iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    cnt = int(row['count_points'])
    mean = row['mean_value']
    p95 = row['p95_value']
    # format numeric succinctly
    def fmt(x):
        try:
            if abs(x) >= 1e6 or (0 < abs(x) < 0.001):
                return f"{x:.4g}"
            return f"{x:.4f}"
        except:
            return str(x)
    lines.append(f"{i+1}) {cmdb} | {kpi} | count={cnt} | mean={fmt(mean)} | p95={fmt(p95)}")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count_points  min_timestamp min_datetime_ASIA_SHANGHAI  max_timestamp max_datetime_ASIA_SHANGHAI    mean_value  median_value     std_value     p95_value
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  7.029032e+01  7.000000e+01  2.116499e+00  7.400000e+01
3         IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  5.793029e+07  5.793029e+07  0.000000e+00  5.793029e+07
2         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  3.921131e+08  3.866685e+08  2.284517e+08  7.572822e+08
1659  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
1660  apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
0         IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  2.040136e+09  2.040136e+09  0.000000e+00  2.040136e+09
1662  apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
15        IG01                             OSLinux-CPU_CPU_CPUWio            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  7.483852e-02  2.080000e-02  9.814109e-02  2.328285e-01
1603  apache02                            OSLinux-CPU_CPU_CPULoad            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  9.677419e-03  0.000000e+00  2.008048e-02  5.500000e-02
1604  apache02                         OSLinux-CPU_CPU_CPUSysTime            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  2.860531e-01  2.907820e-01  6.636712e-02  4.050540e-01
1605  apache02                        OSLinux-CPU_CPU_CPUUserTime            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  1.199177e-01  1.204370e-01  2.854999e-02  1.619800e-01
1606  apache02                             OSLinux-CPU_CPU_CPUWio            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  5.078277e-02  1.250000e-02  1.181902e-01  1.599350e-01
1607  apache02                        OSLinux-CPU_CPU_CPUidleutil            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  9.953696e+01  9.953910e+01  1.463555e-01  9.968020e+01
1593  apache01                 OSLinux-OSLinux_ZABBIX_Host_Uptime            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  6.242393e+06  6.242393e+06  5.458757e+02  6.243203e+06
14        IG01                        OSLinux-CPU_CPU_CPUUserTime            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  2.513662e+01  2.513825e+01  4.619360e-02  2.520630e+01
1586  apache01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  4.935484e-01  5.000000e-01  3.592106e-02  5.000000e-01
1587  apache01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  4.000000e+00  4.000000e+00  0.000000e+00  4.000000e+00
578    Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  3.935484e-01  4.000000e-01  2.497310e-02  4.000000e-01
579    Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  7.000000e+00  7.000000e+00  0.000000e+00  7.000000e+00
585    Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime            31     1615042800  2021-03-06 23:00:00+08:00     1615044600  2021-03-06 23:30:00+08:00  6.242293e+06  6.242293e+06  5.471668e+02  6.243104e+06,    overall_min_timestamp overall_min_datetime_ASIA_SHANGHAI  overall_max_timestamp overall_max_datetime_ASIA_SHANGHAI
0             1615042800          2021-03-06 23:00:00+08:00             1615044600          2021-03-06 23:30:00+08:00)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse existing variables: df_metric, agg_df if present in kernel
# ensure they exist; if not, load metric.csv and recompute agg_df (defensive)
try:
    df_metric
    agg_df
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['timestamp'] = df_metric['timestamp'].astype(int)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    # recompute agg_df global p95 (shouldn't be necessary in normal flow)
    agg_df = (
        df_metric
        .groupby(['cmdb_id','kpi_name'], as_index=False)
        .agg(p95_value = ('value', lambda x: x.quantile(0.95)))
    )

# Time window (inclusive)
start_ts = 1615042800
end_ts = 1615044600

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Filter metric points to the window and store for later inspection
window_metric_df = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge global p95 thresholds from agg_df (precomputed globally) into window data
p95_df = agg_df[['cmdb_id','kpi_name','p95_value']].drop_duplicates()
window_merged = window_metric_df.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

# Mark anomaly points where value >= global p95_value
window_merged['is_anomaly'] = window_merged['value'] >= window_merged['p95_value']

# Aggregate per (cmdb_id, kpi_name) within the window
grp = window_merged.groupby(['cmdb_id','kpi_name'])
agg_window = grp.agg(
    count_window = ('value','count'),
    anomalies_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    p95_value = ('p95_value','first')
).reset_index()

# Earliest anomaly timestamp per group (only among anomaly points)
earliest_anom = (
    window_merged[window_merged['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_anomaly_timestamp'})
)

# Merge earliest anomaly info
agg_window = agg_window.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Convert earliest anomaly timestamp to Asia/Shanghai datetime
agg_window['earliest_anomaly_datetime_ASIA_SHANGHAI'] = pd.to_datetime(
    agg_window['earliest_anomaly_timestamp'], unit='s', utc=True
).dt.tz_convert(tz)

# Compute max_breach_ratio = max_value_in_window / p95_value, handle zero/NaN p95
agg_window['max_breach_ratio'] = agg_window['max_value_in_window'] / agg_window['p95_value']

# Where p95_value is 0 or NaN, set ratio to np.inf if max_value>p95 (i.e., >0), else NaN
zero_p95_mask = (agg_window['p95_value'] == 0) | (agg_window['p95_value'].isna())
agg_window.loc[zero_p95_mask, 'max_breach_ratio'] = np.where(
    agg_window.loc[zero_p95_mask, 'max_value_in_window'] > 0, np.inf, np.nan
)

# Only include rows with anomalies_count >= 1
agg_window = agg_window[agg_window['anomalies_count'] >= 1].copy()

# Select and order columns for compact output
result_cols = [
    'cmdb_id','kpi_name','count_window','anomalies_count',
    'earliest_anomaly_timestamp','earliest_anomaly_datetime_ASIA_SHANGHAI',
    'max_value_in_window','p95_value','max_breach_ratio'
]
agg_window = agg_window.loc[:, result_cols]

# Sort by anomalies_count desc then max_breach_ratio desc, limit to top 20
agg_window = agg_window.sort_values(['anomalies_count','max_breach_ratio'], ascending=[False, False]).head(20)

# Keep full window_metric_df variable in kernel and display compact summary (top table + window size)
agg_window, window_metric_df.shape

```
Out[2]:
```
```python
# Build a concise plain-English summary string based on existing kernel variables:
# uses: agg_window, window_metric_df (already in kernel from previous steps)

import pytz
import pandas as pd

tz = pytz.timezone('Asia/Shanghai')

# Defensive checks (assume variables exist as per prior steps)
try:
    agg_window
    window_metric_df
except NameError:
    raise RuntimeError("Required variables agg_window or window_metric_df not found in kernel.")

# Overall window info
total_points = int(window_metric_df.shape[0])
window_start_ts = int(window_metric_df['timestamp'].min())
window_end_ts = int(window_metric_df['timestamp'].max())
window_start_dt = pd.to_datetime(window_start_ts, unit='s', utc=True).tz_convert(tz)
window_end_dt = pd.to_datetime(window_end_ts, unit='s', utc=True).tz_convert(tz)

# Number of (cmdb_id,kpi_name) series with at least one anomaly in the window
num_series_with_anomaly = int(agg_window.shape[0])

# Prepare top entries (up to 5) for summary
topn = agg_window.head(5).copy()

def fmt_num(x):
    try:
        if pd.isna(x):
            return "NaN"
        x = float(x)
        if x == float('inf'):
            return "inf"
        if abs(x) >= 1e6 or (0 < abs(x) < 0.001):
            return f"{x:.4g}"
        return f"{x:.4f}"
    except:
        return str(x)

lines = []
lines.append("Incident window summary (Asia/Shanghai):")
lines.append(f"- Window timestamps: {window_start_dt} to {window_end_dt} (inclusive).")
lines.append(f"- Total metric points in window: {total_points:,}.")
lines.append(f"- Number of distinct (cmdb_id, kpi_name) with >=1 anomaly: {num_series_with_anomaly}.")
lines.append("- Top anomaly candidates (up to 5), columns: cmdb_id | kpi_name | anomalies_count | earliest_anomaly_dt | max_value | p95_value | max_breach_ratio")
for _, r in topn.iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    anomalies_count = int(r['anomalies_count'])
    ear_dt = r['earliest_anomaly_datetime_ASIA_SHANGHAI']
    if pd.isna(ear_dt):
        ear_dt_str = "N/A"
    else:
        ear_dt_str = pd.to_datetime(ear_dt).tz_convert(tz)
    max_val = fmt_num(r['max_value_in_window'])
    p95 = fmt_num(r['p95_value'])
    ratio = fmt_num(r['max_breach_ratio'])
    lines.append(f"- {cmdb} | {kpi} | anomalies={anomalies_count} | earliest={ear_dt_str} | max={max_val} | p95={p95} | ratio={ratio}")

lines.append("- Note: a few series have p95=0 which yields an infinite breach ratio when max_value>0.")
lines.append("- The full windowed timeseries is stored in variable `window_metric_df`; aggregated results are in `agg_window`.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  count_window  anomalies_count  earliest_anomaly_timestamp earliest_anomaly_datetime_ASIA_SHANGHAI  max_value_in_window     p95_value  max_breach_ratio
573   Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT            31               31                  1615042800               2021-03-06 23:00:00+08:00         1.000000e+00  0.000000e+00               inf
1013  Redis02          redis-Redis_6379_Redis  (keyspace_misses)            31               31                  1615042800               2021-03-06 23:00:00+08:00         2.101000e+04  2.100200e+04          1.000381
0        IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31               31                  1615042800               2021-03-06 23:00:00+08:00         2.040136e+09  2.040136e+09          1.000000
3        IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31               31                  1615042800               2021-03-06 23:00:00+08:00         5.793029e+07  5.793029e+07          1.000000
77       IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31               31                  1615042800               2021-03-06 23:00:00+08:00         1.000000e+00  1.000000e+00          1.000000
80       IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31               31                  1615042800               2021-03-06 23:00:00+08:00         3.000000e-01  3.000000e-01          1.000000
81       IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31               31                  1615042800               2021-03-06 23:00:00+08:00         3.000000e+00  3.000000e+00          1.000000
88       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax            31               31                  1615042800               2021-03-06 23:00:00+08:00         2.040136e+09  2.040136e+09          1.000000
91       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed            31               31                  1615042800               2021-03-06 23:00:00+08:00         5.805949e+07  5.805949e+07          1.000000
165      IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31               31                  1615042800               2021-03-06 23:00:00+08:00         1.000000e+00  1.000000e+00          1.000000
168      IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31               31                  1615042800               2021-03-06 23:00:00+08:00         3.000000e-01  3.000000e-01          1.000000
169      IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31               31                  1615042800               2021-03-06 23:00:00+08:00         3.000000e+00  3.000000e+00          1.000000
176      MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31               31                  1615042800               2021-03-06 23:00:00+08:00         2.040136e+09  2.040136e+09          1.000000
179      MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed            31               31                  1615042800               2021-03-06 23:00:00+08:00         5.068773e+08  5.068773e+08          1.000000
268      MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31               31                  1615042800               2021-03-06 23:00:00+08:00         2.040136e+09  2.040136e+09          1.000000
271      MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed            31               31                  1615042800               2021-03-06 23:00:00+08:00         9.473127e+08  9.473127e+08          1.000000
579   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31               31                  1615042800               2021-03-06 23:00:00+08:00         7.000000e+00  7.000000e+00          1.000000
803   Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31               31                  1615042800               2021-03-06 23:00:00+08:00         1.200000e+01  1.200000e+01          1.000000
880   Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31               31                  1615042800               2021-03-06 23:00:00+08:00         1.000000e+00  1.000000e+00          1.000000
884   Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31               31                  1615042800               2021-03-06 23:00:00+08:00         4.000000e+00  4.000000e+00          1.000000, (28379, 4))```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse agg_window from kernel
tz = pytz.timezone('Asia/Shanghai')

# 1) Basic aggregates by cmdb_id
grp = agg_window.groupby('cmdb_id', as_index=False)

agg_by_cmdb = grp.agg(
    series_with_anomaly = ('kpi_name', 'nunique'),
    total_anomalies = ('anomalies_count', 'sum'),
    earliest_anomaly_timestamp = ('earliest_anomaly_timestamp', 'min'),
    max_breach_ratio = ('max_breach_ratio', 'max')
)

# 2) Convert earliest_anomaly_timestamp to Asia/Shanghai datetime
agg_by_cmdb['earliest_anomaly_datetime_ASIA_SHANGHAI'] = pd.to_datetime(
    agg_by_cmdb['earliest_anomaly_timestamp'], unit='s', utc=True
).dt.tz_convert(tz)

# 3) Find kpi_of_max_breach and its p95 (use idxmax on original agg_window)
idx_of_max = agg_window.groupby('cmdb_id')['max_breach_ratio'].idxmax()
kpi_max_breach = agg_window.loc[idx_of_max, ['cmdb_id','kpi_name','p95_value','max_breach_ratio']].rename(
    columns={'kpi_name':'kpi_of_max_breach','p95_value':'p95_of_kpi_of_max_breach'}
).reset_index(drop=True)

# 4) Merge into final summary
summary_by_cmdb = agg_by_cmdb.merge(
    kpi_max_breach[['cmdb_id','kpi_of_max_breach','p95_of_kpi_of_max_breach']],
    on='cmdb_id',
    how='left'
)

# 5) Order and limit per requirements
summary_by_cmdb = summary_by_cmdb.sort_values(
    by=['series_with_anomaly','max_breach_ratio'],
    ascending=[False, False]
).reset_index(drop=True)

summary_top20 = summary_by_cmdb.head(20)

# 6) Save top 5 cmdb_id by series_with_anomaly into candidates_components
candidates_components = summary_by_cmdb.sort_values(
    by=['series_with_anomaly','max_breach_ratio'],
    ascending=[False, False]
).head(5)['cmdb_id'].astype(str).tolist()

# Display compact results: summary_top20 and candidates_components
summary_top20, candidates_components

```
Out[3]:
```
```python
# Build a concise plain-English summary using existing kernel variables:
# expects summary_top20 and candidates_components to exist from prior steps.

try:
    summary_top20
    candidates_components
except NameError:
    raise RuntimeError("Required variables summary_top20 or candidates_components not found in kernel.")

# Compose summary lines
lines = []
lines.append("Per-component anomaly summary (incident window 2021-03-06 23:00 to 23:30 Asia/Shanghai):")
lines.append(f"- Top components by number of anomalous KPI series:")
for _, r in summary_top20.head(5).iterrows():
    cmdb = r['cmdb_id']
    series_cnt = int(r['series_with_anomaly'])
    total_anoms = int(r['total_anomalies'])
    earliest = r['earliest_anomaly_datetime_ASIA_SHANGHAI']
    max_ratio = r['max_breach_ratio']
    kpi = r['kpi_of_max_breach']
    p95 = r['p95_of_kpi_of_max_breach']
    lines.append(f"  * {cmdb}: {series_cnt} KPI(s) with anomalies, {total_anoms} total anomaly points, earliest={earliest}, top breach KPI={kpi} (p95={p95}, ratio={max_ratio})")

lines.append(f"- Note: Mysql01 shows an infinite breach ratio for a TCP-FIN-WAIT metric because its global p95 is 0 while observed values >0.")
lines.append(f"- Redis02 has a measurable breach on redis keyspace_misses (ratio ~1.00038).")
lines.append(f"- Saved top candidate components for follow-up (by series_with_anomaly): {candidates_components}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id  series_with_anomaly  total_anomalies  earliest_anomaly_timestamp  max_breach_ratio earliest_anomaly_datetime_ASIA_SHANGHAI                              kpi_of_max_breach  p95_of_kpi_of_max_breach
0     IG01                    5              155                  1615042800          1.000000               2021-03-06 23:00:00+08:00       JVM-Memory_7778_JVM_Memory_HeapMemoryMax              2.040136e+09
1     IG02                    5              155                  1615042800          1.000000               2021-03-06 23:00:00+08:00       JVM-Memory_7778_JVM_Memory_HeapMemoryMax              2.040136e+09
2  Mysql01                    2               62                  1615042800               inf               2021-03-06 23:00:00+08:00   OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              0.000000e+00
3     MG01                    2               62                  1615042800          1.000000               2021-03-06 23:00:00+08:00       JVM-Memory_7779_JVM_Memory_HeapMemoryMax              2.040136e+09
4     MG02                    2               62                  1615042800          1.000000               2021-03-06 23:00:00+08:00       JVM-Memory_7779_JVM_Memory_HeapMemoryMax              2.040136e+09
5  Redis01                    2               62                  1615042800          1.000000               2021-03-06 23:00:00+08:00  OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies              1.000000e+00
6  Redis02                    1               31                  1615042800          1.000381               2021-03-06 23:00:00+08:00      redis-Redis_6379_Redis  (keyspace_misses)              2.100200e+04
7  Mysql02                    1               31                  1615042800          1.000000               2021-03-06 23:00:00+08:00     OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount              1.200000e+01, ['IG01', 'IG02', 'Mysql01', 'MG01', 'MG02'])```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Timezone
tz = pytz.timezone('Asia/Shanghai')

# Load trace.csv (reuse df_trace if exists)
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure dtypes
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# 1) Global aggregates per (cmdb_id, trace_name)
agg_trace_global = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        count_points = ('value','count'),
        min_timestamp = ('timestamp','min'),
        max_timestamp = ('timestamp','max'),
        mean_value = ('value','mean'),
        std_value = ('value','std'),
        p95_value = ('value', lambda x: x.quantile(0.95))
    )
)

# 2) Incident window filter (inclusive)
start_ts = 1615042800
end_ts = 1615044600

# Candidate components (use provided list)
candidate_components = ['IG01','IG02','Mysql01','MG01','MG02']

# Window-filtered trace points for candidate components
window_trace_df = df_trace[
    (df_trace['timestamp'] >= start_ts) &
    (df_trace['timestamp'] <= end_ts) &
    (df_trace['cmdb_id'].isin(candidate_components))
].copy()

# 3) Merge global p95 into window data
p95_trace_df = agg_trace_global[['cmdb_id','trace_name','p95_value']].drop_duplicates()
window_merged = window_trace_df.merge(p95_trace_df, on=['cmdb_id','trace_name'], how='left')

# 4) Mark anomalies (value >= global p95)
window_merged['is_anomaly'] = window_merged['value'] >= window_merged['p95_value']

# 5) Aggregate per (cmdb_id, trace_name) within window
grp = window_merged.groupby(['cmdb_id','trace_name'])
agg_window_trace = grp.agg(
    count_window = ('value','count'),
    anomalies_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    p95_value = ('p95_value','first')
).reset_index()

# 6) Earliest anomaly timestamp per group (only among anomaly points)
earliest_anom = (
    window_merged[window_merged['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
    .min()
    .rename(columns={'timestamp':'earliest_anomaly_timestamp'})
)

agg_window_trace = agg_window_trace.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# 7) Convert earliest anomaly timestamp to Asia/Shanghai datetime
agg_window_trace['earliest_anomaly_datetime_ASIA_SHANGHAI'] = pd.to_datetime(
    agg_window_trace['earliest_anomaly_timestamp'], unit='s', utc=True
).dt.tz_convert(tz)

# 8) Compute max_breach_ratio and handle p95==0 or NaN
agg_window_trace['max_breach_ratio'] = agg_window_trace['max_value_in_window'] / agg_window_trace['p95_value']

zero_p95_mask = (agg_window_trace['p95_value'] == 0) | (agg_window_trace['p95_value'].isna())
agg_window_trace.loc[zero_p95_mask, 'max_breach_ratio'] = np.where(
    agg_window_trace.loc[zero_p95_mask, 'max_value_in_window'] > 0, np.inf, np.nan
)

# 9) Keep only series with at least one anomaly
agg_window_trace = agg_window_trace[agg_window_trace['anomalies_count'] >= 1].copy()

# 10) Select and order columns for compact display
result_cols = [
    'cmdb_id','trace_name','count_window','anomalies_count',
    'earliest_anomaly_timestamp','earliest_anomaly_datetime_ASIA_SHANGHAI',
    'max_value_in_window','p95_value','max_breach_ratio'
]
agg_window_trace = agg_window_trace.loc[:, result_cols]

# 11) Sort and limit to top 20
agg_window_trace = agg_window_trace.sort_values(['anomalies_count','max_breach_ratio'], ascending=[False, False]).head(20)

# Final compact outputs: aggregated table (top 20) and stored full windowed traces shape
agg_window_trace, window_trace_df.shape

```
Out[4]:
```
```python
# Summarize the trace analysis results in plain English using existing kernel variables:
# expects agg_window_trace and window_trace_df to be present.

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    agg_window_trace
    window_trace_df
except NameError:
    raise RuntimeError("Required variables agg_window_trace or window_trace_df not found in kernel.")

# Window info
total_trace_points = int(window_trace_df.shape[0])
if total_trace_points > 0:
    ws = int(window_trace_df['timestamp'].min())
    we = int(window_trace_df['timestamp'].max())
    ws_dt = pd.to_datetime(ws, unit='s', utc=True).tz_convert(tz)
    we_dt = pd.to_datetime(we, unit='s', utc=True).tz_convert(tz)
else:
    ws_dt = we_dt = "N/A"

# Top anomalies summary (take up to 6 notable rows)
top_rows = agg_window_trace.head(6).copy().reset_index(drop=True)

lines = []
lines.append("Trace analysis summary (candidate components: IG01, IG02, Mysql01, MG01, MG02):")
lines.append(f"- Window timestamps (Asia/Shanghai): {ws_dt} to {we_dt} (inclusive).")
lines.append(f"- Total trace points for candidates in window: {total_trace_points:,}.")
lines.append(f"- Number of trace series with >=1 anomaly: {agg_window_trace.shape[0]}.")
lines.append("- Top anomalous trace series (up to 6), format: cmdb_id | trace_name | anomalies_count | earliest_anomaly_dt | max_value | p95 | max_breach_ratio:")

def fmt(x):
    try:
        if pd.isna(x):
            return "NaN"
        if x == float('inf'):
            return "inf"
        x = float(x)
        if abs(x) >= 1e6 or (0 < abs(x) < 0.001):
            return f"{x:.4g}"
        return f"{x:.4f}"
    except:
        return str(x)

for _, r in top_rows.iterrows():
    cmdb = r['cmdb_id']
    name = r['trace_name']
    cnt = int(r['anomalies_count'])
    ear_ts = r['earliest_anomaly_timestamp']
    if pd.isna(ear_ts):
        ear_str = "N/A"
    else:
        ear_str = pd.to_datetime(int(ear_ts), unit='s', utc=True).tz_convert(tz)
    maxv = fmt(r['max_value_in_window'])
    p95 = fmt(r['p95_value'])
    ratio = fmt(r['max_breach_ratio'])
    lines.append(f"  * {cmdb} | {name} | anomalies={cnt} | earliest={ear_str} | max={maxv} | p95={p95} | ratio={ratio}")

lines.append("- Observations:")
lines.append("  • MG01 and MG02 show the most and largest trace anomalies in this window, notably traffic row_count metrics (to_docker / from_docker) with multiple breaches.")
lines.append("  • Several duration-related traces (duration_p95 / duration_mean) on MG01/MG02 are highly above their global P95s; the largest breach ratio is MG02 trace.from_dockerA1.duration_p95 (~7.96x) at 2021-03-06 23:24 CST, followed by MG01 trace.from_dockerA1.duration_p95 (~6.83x).")
lines.append("  • Earliest anomalies appear at the window start (2021-03-06 23:00:00+08:00) for multiple series.")
lines.append("  • These results and the full windowed trace points are stored in variables `agg_window_trace` (aggregated) and `window_trace_df` (full points) for follow-up inspection.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                         trace_name  count_window  anomalies_count  earliest_anomaly_timestamp earliest_anomaly_datetime_ASIA_SHANGHAI  max_value_in_window  p95_value  max_breach_ratio
68     MG01        trace.to_dockerB2.row_count            30                4                  1615042800               2021-03-06 23:00:00+08:00            65.000000  48.000000          1.354167
47     MG01      trace.from_dockerA2.row_count            30                4                  1615042860               2021-03-06 23:01:00+08:00            19.000000  15.000000          1.266667
104    MG02        trace.to_dockerB1.row_count            30                3                  1615042920               2021-03-06 23:02:00+08:00            66.000000  55.000000          1.200000
92     MG02      trace.from_dockerB2.row_count            30                3                  1615043100               2021-03-06 23:05:00+08:00            69.000000  59.000000          1.169492
83     MG02      trace.from_dockerA1.row_count            29                3                  1615042980               2021-03-06 23:03:00+08:00            15.000000  13.000000          1.153846
29     IG02        trace.to_Tomcat04.row_count            30                3                  1615042860               2021-03-06 23:01:00+08:00            60.000000  58.000000          1.034483
59     MG01        trace.to_dockerA1.row_count            30                3                  1615043460               2021-03-06 23:11:00+08:00            45.000000  44.000000          1.022727
82     MG02   trace.from_dockerA1.duration_p95            29                2                  1615044240               2021-03-06 23:24:00+08:00             0.267150   0.033560          7.960369
43     MG01   trace.from_dockerA1.duration_p95            29                2                  1615042860               2021-03-06 23:01:00+08:00             0.362000   0.052990          6.831478
81     MG02  trace.from_dockerA1.duration_mean            29                2                  1615044240               2021-03-06 23:24:00+08:00             0.053700   0.011374          4.721128
42     MG01  trace.from_dockerA1.duration_mean            29                2                  1615042860               2021-03-06 23:01:00+08:00             0.071000   0.021833          3.251928
88     MG02   trace.from_dockerB1.duration_p95            30                2                  1615043100               2021-03-06 23:05:00+08:00             0.785800   0.261275          3.007559
58     MG01     trace.to_dockerA1.duration_p95            30                2                  1615043340               2021-03-06 23:09:00+08:00             1.864900   0.648710          2.874782
85     MG02   trace.from_dockerA2.duration_p95            29                2                  1615043040               2021-03-06 23:04:00+08:00             0.258900   0.097610          2.652392
49     MG01   trace.from_dockerB1.duration_p95            30                2                  1615043760               2021-03-06 23:16:00+08:00             0.197800   0.075167          2.631456
57     MG01    trace.to_dockerA1.duration_mean            30                2                  1615043340               2021-03-06 23:09:00+08:00             0.256741   0.116490          2.203977
37     MG01   trace.from_Tomcat03.duration_p95            30                2                  1615042800               2021-03-06 23:00:00+08:00             1.763800   0.918117          1.921105
84     MG02  trace.from_dockerA2.duration_mean            29                2                  1615043040               2021-03-06 23:04:00+08:00             0.052000   0.027451          1.894271
36     MG01  trace.from_Tomcat03.duration_mean            30                2                  1615043340               2021-03-06 23:09:00+08:00             0.226372   0.133502          1.695645
99     MG02    trace.to_dockerA2.duration_mean            30                2                  1615043460               2021-03-06 23:11:00+08:00             0.233375   0.139008          1.678855, (3231, 4))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Time window
start_ts = 1615042800
end_ts = 1615044600

# Candidate list for these inspections (includes Redis02 per instruction)
candidate_list = ['IG01','IG02','Mysql01','MG01','MG02','Redis02']

# Timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) error_logs.csv processing
df_error_logs = pd.read_csv("error_logs.csv")
df_error_logs['timestamp'] = df_error_logs['timestamp'].astype(int)
# Filter to window and candidate components
window_error_logs_df = df_error_logs[
    (df_error_logs['timestamp'] >= start_ts) &
    (df_error_logs['timestamp'] <= end_ts) &
    (df_error_logs['cmdb_id'].isin(candidate_list))
].copy()

# Aggregate per cmdb_id: message_count, earliest_message_timestamp, up to 3 distinct sample messages
if not window_error_logs_df.empty:
    err_grp = (
        window_error_logs_df
        .groupby('cmdb_id', as_index=False)
        .agg(
            message_count = ('message','count'),
            earliest_message_timestamp = ('timestamp','min'),
            sample_messages = ('message', lambda s: ' || '.join(pd.Series(s.dropna().astype(str).unique()[:3])))
        )
    )
    err_grp['earliest_message_datetime_ASIA_SHANGHAI'] = pd.to_datetime(
        err_grp['earliest_message_timestamp'], unit='s', utc=True
    ).dt.tz_convert(tz)
    window_error_logs_df = err_grp.sort_values('message_count', ascending=False).head(20).reset_index(drop=True)
else:
    # empty result frame with desired columns
    window_error_logs_df = pd.DataFrame(columns=[
        'cmdb_id','message_count','earliest_message_timestamp','sample_messages','earliest_message_datetime_ASIA_SHANGHAI'
    ])

# 2) log.csv processing
df_log = pd.read_csv("log.csv")
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Compute global P95 per (cmdb_id, log_name) using full file (rule 9)
p95_log = (
    df_log
    .groupby(['cmdb_id','log_name'], as_index=False)
    .agg(p95_value = ('value', lambda x: x.quantile(0.95)))
)

# Filter df_log to window and candidate components
window_log_points = df_log[
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts) &
    (df_log['cmdb_id'].isin(candidate_list))
].copy()

# Merge global p95 into windowed points
window_log_merged = window_log_points.merge(p95_log, on=['cmdb_id','log_name'], how='left')

# Mark anomalies where value >= p95_value
window_log_merged['is_anomaly'] = window_log_merged['value'] >= window_log_merged['p95_value']

# Aggregate per (cmdb_id, log_name)
if not window_log_merged.empty:
    agg_log = (
        window_log_merged
        .groupby(['cmdb_id','log_name'], as_index=False)
        .agg(
            count_window = ('value','count'),
            sum_value_in_window = ('value','sum'),
            anomalies_count = ('is_anomaly','sum'),
            p95_value = ('p95_value','first')
        )
    )

    # earliest anomaly timestamp per group (only among anomaly points)
    earliest_anom = (
        window_log_merged[window_log_merged['is_anomaly']]
        .groupby(['cmdb_id','log_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anomaly_timestamp'})
    )
    agg_log = agg_log.merge(earliest_anom, on=['cmdb_id','log_name'], how='left')

    # Convert earliest anomaly timestamp to Asia/Shanghai datetime
    agg_log['earliest_anomaly_datetime_ASIA_SHANGHAI'] = pd.to_datetime(
        agg_log['earliest_anomaly_timestamp'], unit='s', utc=True
    ).dt.tz_convert(tz)

    # Flag log_name containing 'error' or 'err' (case-insensitive) and sum_value_in_window>0
    agg_log['log_name_lower'] = agg_log['log_name'].str.lower()
    agg_log['error_flag'] = ((agg_log['log_name_lower'].str.contains('error')) | (agg_log['log_name_lower'].str.contains('err'))) & (agg_log['sum_value_in_window'] > 0)
    agg_log = agg_log.drop(columns=['log_name_lower'])

    # Sort by anomalies_count desc then sum_value_in_window desc and limit to top 20
    window_log_agg_df = agg_log.sort_values(['anomalies_count','sum_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    window_log_agg_df = pd.DataFrame(columns=[
        'cmdb_id','log_name','count_window','sum_value_in_window','anomalies_count',
        'p95_value','earliest_anomaly_timestamp','earliest_anomaly_datetime_ASIA_SHANGHAI','error_flag'
    ])

# Final compact outputs (stored in kernel variables): window_error_logs_df, window_log_agg_df
window_error_logs_df, window_log_agg_df

```
Out[5]:
```
# Build a concise plain-English summary using existing kernel variables:
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    window_error_logs_df
    window_log_agg_df
except NameError:
    raise RuntimeError("Required variables window_error_logs_df or window_log_agg_df not found in kernel.")

lines = []
lines.append("Logs inspection summary for incident window 2021-03-06 23:00:00 to 23:30:00 (Asia/Shanghai):")

# Part 1: error_logs.csv
if window_error_logs_df.shape[0] == 0:
    lines.append("- error_logs.csv: No error log entries found for candidate components (IG01, IG02, Mysql01, MG01, MG02, Redis02) in the window.")
else:
    lines.append("- error_logs.csv (per component):")
    for _, r in window_error_logs_df.iterrows():
        cmdb = r['cmdb_id']
        cnt = int(r['message_count'])
        ts = int(r['earliest_message_timestamp'])
        dt = pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz)
        samples = r['sample_messages']
        lines.append(f"  * {cmdb}: {cnt} messages, earliest={dt}, samples={samples}")

# Part 2: log.csv aggregated
if window_log_agg_df.shape[0] == 0:
    lines.append("- log.csv: No log series for candidate components in the window.")
else:
    lines.append("- log.csv aggregated (per cmdb_id, log_name) with global P95-based anomalies:")
    for _, r in window_log_agg_df.iterrows():
        cmdb = r['cmdb_id']
        name = r['log_name']
        cnt = int(r['count_window'])
        ssum = float(r['sum_value_in_window'])
        anoms = int(r['anomalies_count'])
        p95 = r['p95_value']
        ear_ts = r['earliest_anomaly_timestamp']
        ear_dt = pd.to_datetime(int(ear_ts), unit='s', utc=True).tz_convert(tz) if pd.notna(ear_ts) else "N/A"
        err_flag = bool(r.get('error_flag', False))
        lines.append(f"  * {cmdb} | {name} | points={cnt} | sum={ssum} | anomalies={anoms} | p95={p95} | earliest_anomaly={ear_dt} | error_flag={err_flag}")

    lines.append("  Note: error_flag=True indicates log_name contains 'error' or 'err' and sum_value_in_window>0.")

# Observations / actions
lines.append("- Observations:")
lines.append("  • No raw error log messages found for the inspected components in the window.")
lines.append("  • Only Mysql01 shows log-series anomalies in the window:")
lines.append("      - log.row_count: 10 points, sum=10, all 10 points exceed global P95 (p95=1).")
lines.append("      - log.error_count: 10 points, sum=0, anomalies_count=10 but sum=0 (p95=0); error_flag is False because sum_value_in_window==0.")
lines.append("  • Save locations: window_error_logs_df (error logs summary) and window_log_agg_df (log aggregates) for follow-up inspection.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, message_count, earliest_message_timestamp, sample_messages, earliest_message_datetime_ASIA_SHANGHAI]
Index: [],    cmdb_id         log_name  count_window  sum_value_in_window  anomalies_count  p95_value  earliest_anomaly_timestamp earliest_anomaly_datetime_ASIA_SHANGHAI  error_flag
0  Mysql01    log.row_count            10                 10.0               10        1.0                  1615042800               2021-03-06 23:00:00+08:00       False
1  Mysql01  log.error_count            10                  0.0               10        0.0                  1615042800               2021-03-06 23:00:00+08:00       False)```
```